# Short Volshock Fold-Local WF2020 Target Manifest v1

This notebook consumes the ready reward/checkpoint recipe and merges the existing `wf_2020_train` SOXLBacktest-derived target contract with the existing short-volshock fold-local targets for `wf_2022_train` and `wf_2023_2024_train`.

Purpose: unblock the next **BC slice screen** from a target-coverage perspective while keeping official replay, PPO, promotion, and actor retention blocked until strict walk-forward evidence exists.

In [1]:
from __future__ import annotations

import json
import sys
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "SPEC.md").exists())
for candidate in (ROOT, ROOT / "src"):
    text = str(candidate)
    if text not in sys.path:
        sys.path.insert(0, text)

from src.soxl_quant.app.reward_checkpoint_screen import audit_reward_checkpoint_recipe_scope

EXPERIMENT = "short_volshock_foldlocal_wf2020_target_manifest_v1"
OUT_DIR = ROOT / "runtime" / "model_selection" / "card" / EXPERIMENT
OUT_DIR.mkdir(parents=True, exist_ok=True)

PATHS = {
    "contract": ROOT / "runtime/reward/card/short_volshock_teacher_reward_checkpoint_screen_v1/reward_checkpoint_recipe_contract.json",
    "selected_official_replay_fold_metrics": ROOT / "runtime/model_selection/card/short_volshock_official_replay_screen_v1/selected_official_replay_fold_metrics.csv",
    "wf2020_target_contract": ROOT / "runtime/model_selection/card/cap10_wf2020_soxlbacktest_target_contract_v1/soxlbacktest_wf2020_target_contract.csv",
    "wf2020_scope_audit": ROOT / "runtime/model_selection/card/cap10_wf2020_soxlbacktest_target_contract_v1/target_scope_audit.csv",
    "short_volshock_targets": ROOT / "runtime/model_selection/card/short_volshock_target_generation_handoff_v1/short_volshock_de_risk_targets.csv",
    "short_volshock_scope_audit": ROOT / "runtime/model_selection/card/short_volshock_target_generation_handoff_v1/target_scope_audit.csv",
}

missing = [name for name, path in PATHS.items() if not path.exists()]
if missing:
    raise FileNotFoundError(f"missing source artifacts: {missing}")

source_manifest = pd.DataFrame([
    {"source_label": name, "path": str(path.relative_to(ROOT)), "exists": path.exists()}
    for name, path in PATHS.items()
])
source_manifest.to_csv(OUT_DIR / "source_manifest.csv", index=False)
source_manifest

,source_label,path,exists
0,contract,runtime/reward/card/short_volshock_teacher_rew...,True
1,selected_official_replay_fold_metrics,runtime/model_selection/card/short_volshock_of...,True
2,wf2020_target_contract,runtime/model_selection/card/cap10_wf2020_soxl...,True
3,wf2020_scope_audit,runtime/model_selection/card/cap10_wf2020_soxl...,True
4,short_volshock_targets,runtime/model_selection/card/short_volshock_ta...,True
5,short_volshock_scope_audit,runtime/model_selection/card/short_volshock_ta...,True


In [2]:
contract = json.loads(PATHS["contract"].read_text(encoding="utf-8"))
selected_fold_metrics = pd.read_csv(PATHS["selected_official_replay_fold_metrics"])
selected_fold_metrics = selected_fold_metrics.copy()
selected_fold_metrics["teacher_usage"] = "official_replay_reference_only_not_training_label"
selected_fold_metrics["teacher_status"] = selected_fold_metrics["status"].astype(str)

required_windows = contract["final_required_windows"]
contract_missing_windows = contract.get("missing_final_required_windows", [])
print("required_windows", required_windows)
print("contract_missing_windows", contract_missing_windows)
print(selected_fold_metrics[["fold", "teacher_usage", "teacher_status", "full_gate", "official_warm_relative_return", "official_warm_max_dd"]].to_string(index=False))

required_windows ['wf_2020', 'wf_2022', 'wf_2023_2024']
contract_missing_windows ['wf_2020']
        fold                                     teacher_usage teacher_status  full_gate  official_warm_relative_return  official_warm_max_dd
     wf_2022 official_replay_reference_only_not_training_label      evaluated       True                      83.564183              8.746809
wf_2023_2024 official_replay_reference_only_not_training_label      evaluated      False                      29.817263             15.107381


In [3]:
def final_window_key(value: object) -> str:
    text = str(value or "").strip()
    for suffix in ("_train", "_target", "_eval", "_test"):
        if text.endswith(suffix):
            return text[: -len(suffix)]
    return text


def manifest_from_scope(scope_path: Path, target_path: Path, source_label: str) -> pd.DataFrame:
    scope = pd.read_csv(scope_path)
    rows = []
    for row in scope.to_dict("records"):
        fold = str(row.get("fold") or row.get("target_fold") or row.get("window"))
        rows.append({
            "fold": fold,
            "window": str(row.get("window") or fold),
            "target_fold": str(row.get("target_fold") or fold),
            "final_window": final_window_key(fold),
            "target_path": str(target_path.relative_to(ROOT)),
            "eval_overlap_rows": int(row.get("eval_overlap_rows", 0)),
            "outside_train_eval_rows": int(row.get("outside_train_eval_rows", 0)),
            "target_scope": str(row.get("target_scope", "fold_local")),
            "train_start": str(row.get("train_start", "")),
            "train_end": str(row.get("train_end", "")),
            "eval_start": str(row.get("eval_start", "")),
            "eval_end": str(row.get("eval_end", "")),
            "source_label": source_label,
        })
    return pd.DataFrame(rows)

fold_local_target_manifest = pd.concat(
    [
        manifest_from_scope(PATHS["wf2020_scope_audit"], PATHS["wf2020_target_contract"], "wf2020_soxlbacktest_target_contract"),
        manifest_from_scope(PATHS["short_volshock_scope_audit"], PATHS["short_volshock_targets"], "short_volshock_target_generation_handoff"),
    ],
    ignore_index=True,
)
fold_local_target_manifest = fold_local_target_manifest.sort_values(["final_window", "fold"]).reset_index(drop=True)
fold_local_target_manifest.to_csv(OUT_DIR / "fold_local_target_manifest.csv", index=False)
fold_local_target_manifest

,fold,window,target_fold,final_window,target_path,eval_overlap_rows,outside_train_eval_rows,target_scope,train_start,train_end,eval_start,eval_end,source_label
0,wf_2020_train,wf_2020_train,wf_2020_train,wf_2020,runtime/model_selection/card/cap10_wf2020_soxl...,0,0,fold_local,2010-03-11,2019-12-31,2020-01-01,2020-12-31,wf2020_soxlbacktest_target_contract
1,wf_2022_train,wf_2022_train,wf_2022_train,wf_2022,runtime/model_selection/card/short_volshock_ta...,0,0,fold_local,2020-01-02,2021-12-31,2022-01-03,2022-12-30,short_volshock_target_generation_handoff
2,wf_2023_2024_train,wf_2023_2024_train,wf_2023_2024_train,wf_2023_2024,runtime/model_selection/card/short_volshock_ta...,0,0,fold_local,2020-01-02,2022-12-30,2023-01-03,2024-12-31,short_volshock_target_generation_handoff


In [4]:
baseline_audit = audit_reward_checkpoint_recipe_scope(
    contract,
    teacher_window_coverage=pd.DataFrame(),
    selected_fold_metrics=selected_fold_metrics,
    fold_local_target_manifest=None,
)
after_audit = audit_reward_checkpoint_recipe_scope(
    contract,
    teacher_window_coverage=pd.DataFrame(),
    selected_fold_metrics=selected_fold_metrics,
    fold_local_target_manifest=fold_local_target_manifest,
)

for name, frame in [
    ("coverage_audit_before_targets.csv", baseline_audit["coverage_rows"]),
    ("scope_audit_before_targets.csv", baseline_audit["scope_rows"]),
    ("required_gate_rows_before_targets.csv", baseline_audit["required_gate_rows"]),
    ("coverage_audit_after_targets.csv", after_audit["coverage_rows"]),
    ("scope_audit_after_targets.csv", after_audit["scope_rows"]),
    ("required_gate_rows_after_targets.csv", after_audit["required_gate_rows"]),
]:
    frame.to_csv(OUT_DIR / name, index=False)

print("before", baseline_audit["summary"])
print("after", after_audit["summary"])
after_audit["coverage_rows"]

before {'decision': 'recipe_scope_audit_blocks_training', 'selected_candidate': 'teacher_short_volshock_cap05_hit2', 'training_allowed': False, 'promotion_allowed': False, 'coverage_gate': False, 'official_replay_coverage_gate': False, 'target_coverage_gate': False, 'scope_gate': False, 'missing_final_required_windows': ['wf_2020'], 'official_missing_final_required_windows': ['wf_2020'], 'required_window_count': 3, 'target_ready_window_count': 0}
after {'decision': 'recipe_scope_audit_training_ready', 'selected_candidate': 'teacher_short_volshock_cap05_hit2', 'training_allowed': True, 'promotion_allowed': False, 'coverage_gate': True, 'official_replay_coverage_gate': False, 'target_coverage_gate': True, 'scope_gate': True, 'missing_final_required_windows': [], 'official_missing_final_required_windows': ['wf_2020'], 'required_window_count': 3, 'target_ready_window_count': 3}


,window,status,present,official_present,target_present,teacher_status,full_gate,daily_path
0,wf_2020,target_ready,True,False,True,,False,
1,wf_2022,present,True,True,True,evaluated,True,runtime/model_selection/card/short_volshock_of...
2,wf_2023_2024,present,True,True,True,evaluated,False,runtime/model_selection/card/short_volshock_of...


In [5]:
summary = dict(after_audit["summary"])
summary.update({
    "experiment": EXPERIMENT,
    "created_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "decision": "foldlocal_target_coverage_training_ready_official_replay_still_blocks_promotion"
    if after_audit["summary"]["training_allowed"] and not after_audit["summary"]["official_replay_coverage_gate"]
    else after_audit["summary"]["decision"],
    "reference_profile": contract.get("reference_profile"),
    "selected_candidate": contract.get("selected_candidate"),
    "bc_screen_allowed_next": bool(after_audit["summary"]["training_allowed"]),
    "ppo_allowed_next": False,
    "actor_retention_allowed": False,
    "final_required_windows": required_windows,
    "source_contract": str(PATHS["contract"].relative_to(ROOT)),
    "target_manifest_path": str((OUT_DIR / "fold_local_target_manifest.csv").relative_to(ROOT)),
    "next_mechanism": "run_three_fold_short_volshock_bc_slice_screen_before_any_ppo",
    "promotion_blocker": "wf_2020 official replay coverage and strict final walk-forward retention gate are still required before promotion.",
    "sidecars": [
        "source_manifest.csv",
        "fold_local_target_manifest.csv",
        "coverage_audit_before_targets.csv",
        "scope_audit_before_targets.csv",
        "required_gate_rows_before_targets.csv",
        "coverage_audit_after_targets.csv",
        "scope_audit_after_targets.csv",
        "required_gate_rows_after_targets.csv",
        "selection.json",
        "summary.json",
        "conclusion.md",
    ],
})

selection = {
    "experiment": EXPERIMENT,
    "decision": summary["decision"],
    "selected_candidate": summary["selected_candidate"],
    "target_ready_windows": after_audit["scope_rows"].loc[after_audit["scope_rows"]["target_ready"], "fold"].astype(str).tolist(),
    "official_missing_final_required_windows": summary["official_missing_final_required_windows"],
    "next_action": summary["next_mechanism"],
}

(OUT_DIR / "summary.json").write_text(json.dumps(summary, indent=2, sort_keys=True) + "\n", encoding="utf-8")
(OUT_DIR / "selection.json").write_text(json.dumps(selection, indent=2, sort_keys=True) + "\n", encoding="utf-8")
conclusion = "\n".join([
    f"# {EXPERIMENT}",
    "",
    f"- decision: `{summary['decision']}`",
    f"- training_allowed: `{summary['training_allowed']}`; bc_screen_allowed_next: `{summary['bc_screen_allowed_next']}`",
    f"- target_coverage_gate: `{summary['target_coverage_gate']}`; target_ready_window_count: `{summary['target_ready_window_count']}/{summary['required_window_count']}`",
    f"- official_replay_coverage_gate: `{summary['official_replay_coverage_gate']}`; official_missing_final_required_windows: `{','.join(summary['official_missing_final_required_windows'])}`",
    "- promotion_allowed, ppo_allowed_next, and actor_retention_allowed remain false.",
    f"- next_action: `{summary['next_mechanism']}`",
])
(OUT_DIR / "conclusion.md").write_text(conclusion + "\n", encoding="utf-8")

assert summary["training_allowed"] is True
assert summary["bc_screen_allowed_next"] is True
assert summary["target_coverage_gate"] is True
assert summary["target_ready_window_count"] == 3
assert summary["official_replay_coverage_gate"] is False
assert summary["promotion_allowed"] is False
assert summary["ppo_allowed_next"] is False
assert summary["actor_retention_allowed"] is False
summary

{'decision': 'foldlocal_target_coverage_training_ready_official_replay_still_blocks_promotion',
 'selected_candidate': 'teacher_short_volshock_cap05_hit2',
 'training_allowed': True,
 'promotion_allowed': False,
 'coverage_gate': True,
 'official_replay_coverage_gate': False,
 'target_coverage_gate': True,
 'scope_gate': True,
 'missing_final_required_windows': [],
 'official_missing_final_required_windows': ['wf_2020'],
 'required_window_count': 3,
 'target_ready_window_count': 3,
 'experiment': 'short_volshock_foldlocal_wf2020_target_manifest_v1',
 'created_at': '2026-06-16T06:54:45+00:00',
 'reference_profile': 'soxl_backtest_policy_reimplementation_v1',
 'bc_screen_allowed_next': True,
 'ppo_allowed_next': False,
 'actor_retention_allowed': False,
 'final_required_windows': ['wf_2020', 'wf_2022', 'wf_2023_2024'],
 'source_contract': 'runtime/reward/card/short_volshock_teacher_reward_checkpoint_screen_v1/reward_checkpoint_recipe_contract.json',
 'target_manifest_path': 'runtime/mode

## Conclusion

This run merges all three required fold-local training target windows: `wf_2020`, `wf_2022`, and `wf_2023_2024`. The scope audit now allows the next BC slice screen because the target coverage and zero-eval-overlap gates pass.

This is not a promotion. `wf_2020` official replay coverage and the strict final walk-forward retention gate are still blocked, so PPO continuation, actor retention, and final model promotion remain false. The next action is to run the three-fold short-volshock BC slice screen before any PPO.